In [ ]:
# Rutas del proyecto (INPUT / OUTPUT)
function _resolver_base()
    for start in unique([(@__DIR__), pwd()])
        d = start
        for _ in 1:5
            isdir(joinpath(d, "INPUT", "24 NODE MODEL")) && return d
            nd = dirname(d); nd == d && break; d = nd
        end
    end
    return dirname(dirname(@__DIR__))
end
const BASE    = _resolver_base()
const IN_BESS = joinpath(BASE, "INPUT", "BESS")
const IN_WIND = joinpath(BASE, "INPUT", "Wind power generation forecast")
const IN_NODE = joinpath(BASE, "INPUT", "24 NODE MODEL")
const OUT_NB  = joinpath(BASE, "OUTPUT", "NOTEBOOKS")
for d in ["Basic","Eolico","BESS","Complete","UC"]; mkpath(joinpath(OUT_NB, d)); end
println("BASE del proyecto: ", BASE)


In [ ]:
#-------------------------LIBRERIAS-----------------------------
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX


#-------------------------DATOS-----------------------------

# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]

buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1


# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]

# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])

T = length(demanda_horaria) # Longitud del horizonte temporal


# PARÁMETROS EÓLICOS
# Archivo Datos Eolicos
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]

# Detectar cuántos parques eólicos hay 
nEol = count(!ismissing, sheet_eolico["B"][2:end])

# Leer vectores por parque eólico
bus_eolico = Int(sheet_eolico["A"][2])
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2])
A_vec = Float64.(sheet_eolico["D"][2])
Cp_vec = Float64.(sheet_eolico["E"][2])

# PARÁMETROS VIENTO
# Archivo de Viento
sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]

velocidades_viento = Float64.(sheet_viento["B"][2:end])
function potencia_eolica_teorica(v)
    if v < 3 || v > 25
        return 0.0
    elseif v <= 11.8
        return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
    else
        return 1.5
    end
end
potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]

# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
sheet_lineas = xlsx_lineas[1]

# Detectar cuántas líneas hay 
nLineas = count(!ismissing, sheet_lineas["A"][2:end])

# Leer las columnas desde fila 2 hacia abajo
desde = Int64.(sheet_lineas["A"][2:1+nLineas])
hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])

# Crear diccionarios para reactancia y Fmax
x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
Sbase = 100.0 # Potencia base del sistema en MVA

nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema

# PARÁMETROS DE BESS
# Leer el archivo y hoja
xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
bus_bess         = Int(sheet_bess["A"][2])
Emax             = Float64(sheet_bess["B"][2])
pev_max          = Float64(sheet_bess["C"][2])
η_c              = Float64(sheet_bess["D"][2])
η_d              = Float64(sheet_bess["E"][2])
ramp_pev         = Float64(sheet_bess["F"][2])
SOC_ini          = Float64(sheet_bess["G"][2])
SOC_min_pct      = Float64(sheet_bess["H"][2])
pen_soc_low      = Float64(sheet_bess["I"][2])
carga_bess       = Float64(sheet_bess["J"][2])
descarga_bess    = Float64(sheet_bess["K"][2])
degradacion_bess = Float64(sheet_bess["L"][2])


# DISTRIBUCION DE CARGA 
xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
sheet_carga = xlsx_carga[1]

# Leer columnas
buses_carga = Int64.(sheet_carga["A"][2:end])
porcentajes_carga = Float64.(sheet_carga["B"][2:end])

# Crear un diccionario para acceso rápido
distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))


#-------------------------MODELO-----------------------------
model = Model(HiGHS.Optimizer)


#-------------------------VARIABLES-----------------------------
# VARIABLES
#@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
@variable(model, p[g=1:nGen, t=1:T])
@variable(model, u[g=1:nGen, t=1:T], Bin)
@variable(model, v[g=1:nGen, t=1:T], Bin)
@variable(model, θ[b=1:nBuses, t=1:T])
@variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
@variable(model, 0 <= charge[t=1:T] <= pev_max)
@variable(model, 0 <= discharge[t=1:T] <= pev_max)
@variable(model, 0 <= soc[t=1:T] <= Emax)
@variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
@variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga


#-------------------------RESTRICCIONES-----------------------------


#----------RESTRICCIONES DE BALANCE DE NODOS-------
# BALANCE NODAL
for t in 1:T
    for b in 1:nBuses
        gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
        gen += (b == bus_eolico ? pw[t] : 0.0)
        gen += (b == bus_bess ? discharge[t] : 0.0)
        
        demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
        if b == bus_bess
            demanda_b += charge[t]
        end

        inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
        outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)

        @constraint(model, gen + inflow - outflow == demanda_b)
    end
end

#----------RESTRICCIONES DE GENERADORES-------
# RESTRICCIONES OPERATIVAS GENERADORES
for g in 1:nGen, t in 1:T
    @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
    @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
end

# RESTRICCIONES DE ARRANQUE
for g in 1:nGen
    @constraint(model, v[g,1] >= u[g,1] - u0[g])
    @constraint(model, v[g,1] <= u[g,1])
    @constraint(model, v[g,1] <= 1 - u0[g])
    for t in 2:T
        @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
        @constraint(model, v[g,t] <= u[g,t])
        @constraint(model, v[g,t] <= 1 - u[g,t-1])
    end
end

#----------RESTRICCIONES DEL BESS-------
# SOC DINÁMICO Y RESTRICCIONES DEL BESS
@constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
for t in 2:T
    @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
    @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
    @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
    @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
    @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
end

# Penalización por SOC bajo
for t in 1:T
    @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
end

#Evitar cargar y descargar el bess al mismo tiempo 
for t in 1:T
    @constraint(model, charge[t] <= pev_max * (1 - z[t]))
    @constraint(model, discharge[t] <= pev_max * z[t])
end

#----------RESTRICCIONES DE REDES-------
# LÍMITES DE FLUJO
for (i,j) in keys(x), t in 1:T
    fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
    @constraint(model, fij <= Fmax[(i,j)])
    @constraint(model, fij >= -Fmax[(i,j)])
end


# ÁNGULO DE REFERENCIA
for t in 1:T
    @constraint(model, θ[1,t] == 0)
end

# OBJETIVO
# CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
@constraint(model, soc[T] >= SOC_ini / 100 * Emax)

@objective(model, Min,
    sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
    sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) +
    sum(pen_low[t]*pen_soc_low for t=1:T)
    + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
    + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
)

# OPTIMIZACIÓN
tic = time()
optimize!(model)
toc = time()

# RESULTADOS
status = termination_status(model)
println("\nEstado de optimización: ", status)
println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

if status == MOI.OPTIMAL || status == MOI.FEASIBLE_POINT
    println("\nResumen horario de operación:")
    # Encabezado dinámico
    print("Hora\tDemanda")
    for g in 1:nGen
        print("\tG$g")
    end
    println("\tEólica\tCharge\tDischarge\tSOC")
    
    # Filas de resultados
    for t in 1:T
        gvals = [round(value(p[g,t]), digits=2) for g in 1:nGen]
        print("$t\t", round(demanda_horaria[t], digits=2))
        for val in gvals
            print("\t", val)
        end
        println("\t", round(value(pw[t]), digits=2), "\t",
                round(value(charge[t]), digits=2), "\t",
                round(value(discharge[t]), digits=2), "\t",
                round(value(soc[t]), digits=2))
    end
    println("\nCosto total: ", round(objective_value(model), digits=2))
else
    println("\nNo se encontró solución factible.")
end


In [ ]:
#-------------------------LIBRERIAS-----------------------------
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX

 
#-------------------------DATOS-----------------------------
        
# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
     
buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1

# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
        
# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])
        
T = length(demanda_horaria) # Longitud del horizonte temporal
        
        
# PARÁMETROS EÓLICOS
# Archivo Datos Eolicos
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
       
# Leer vectores por parque eólico
        
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2])
A_vec = Float64.(sheet_eolico["D"][2])
Cp_vec = Float64.(sheet_eolico["E"][2])


# PARÁMETROS VIENTO
# Archivo de Viento
        sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
        
        velocidades_viento = Float64.(sheet_viento["B"][2:end])
        function potencia_eolica_teorica(v)
            if v < 3 || v > 25
                return 0.0
            elseif v <= 11.8
                return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
            else
                return 1.5
            end
        end
        potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]
        
# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
        xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
        sheet_lineas = xlsx_lineas[1]
        
# Detectar cuántas líneas hay 
        nLineas = count(!ismissing, sheet_lineas["A"][2:end])
        
# Leer las columnas desde fila 2 hacia abajo
        desde = Int64.(sheet_lineas["A"][2:1+nLineas])
        hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
        reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
        flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
        
# Crear diccionarios para reactancia y Fmax
        x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
        Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
        Sbase = 100.0 # Potencia base del sistema en MVA
        
        nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema
        
# PARÁMETROS DE BESS
# Leer el archivo y hoja
        xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
        sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
        
        Emax             = Float64(sheet_bess["B"][2])
        pev_max          = Float64(sheet_bess["C"][2])
        η_c              = Float64(sheet_bess["D"][2])
        η_d              = Float64(sheet_bess["E"][2])
        ramp_pev         = Float64(sheet_bess["F"][2])
        SOC_ini          = Float64(sheet_bess["G"][2])
        SOC_min_pct      = Float64(sheet_bess["H"][2])
        pen_soc_low      = Float64(sheet_bess["I"][2])
        carga_bess       = Float64(sheet_bess["J"][2])
        descarga_bess    = Float64(sheet_bess["K"][2])
        degradacion_bess = Float64(sheet_bess["L"][2])
        
# DISTRIBUCION DE CARGA 
        xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
        sheet_carga = xlsx_carga[1]
        
        # Leer columnas
        buses_carga = Int64.(sheet_carga["A"][2:end])
        porcentajes_carga = Float64.(sheet_carga["B"][2:end])
        
        # Crear un diccionario para acceso rápido
        distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))        
        



function evaluar_modelo(bus_bess_actual::Int, bus_eolico_actual::Int)
        bus_eolico = bus_eolico_actual
        bus_bess         = bus_bess_actual        
        
        #-------------------------MODELO-----------------------------
        model = Model(HiGHS.Optimizer)
        
        
        #-------------------------VARIABLES-----------------------------
        # VARIABLES
        #@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
        @variable(model, p[g=1:nGen, t=1:T])
        @variable(model, u[g=1:nGen, t=1:T], Bin)
        @variable(model, v[g=1:nGen, t=1:T], Bin)
        @variable(model, θ[b=1:nBuses, t=1:T])
        @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
        @variable(model, 0 <= charge[t=1:T] <= pev_max)
        @variable(model, 0 <= discharge[t=1:T] <= pev_max)
        @variable(model, 0 <= soc[t=1:T] <= Emax)
        @variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
        @variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga
        
        
        #-------------------------RESTRICCIONES-----------------------------
        
        
        #----------RESTRICCIONES DE BALANCE DE NODOS-------
        # BALANCE NODAL
        for t in 1:T
            for b in 1:nBuses
                gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
                gen += (b == bus_eolico ? pw[t] : 0.0)
                gen += (b == bus_bess ? discharge[t] : 0.0)
                
                demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
                if b == bus_bess
                    demanda_b += charge[t]
                end
        
                inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
                outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        
                @constraint(model, gen + inflow - outflow == demanda_b)
            end
        end
        
        #----------RESTRICCIONES DE GENERADORES-------
        # RESTRICCIONES OPERATIVAS GENERADORES
        for g in 1:nGen, t in 1:T
            @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
            @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
        end
        
        # RESTRICCIONES DE ARRANQUE
        for g in 1:nGen
            @constraint(model, v[g,1] >= u[g,1] - u0[g])
            @constraint(model, v[g,1] <= u[g,1])
            @constraint(model, v[g,1] <= 1 - u0[g])
            for t in 2:T
                @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
                @constraint(model, v[g,t] <= u[g,t])
                @constraint(model, v[g,t] <= 1 - u[g,t-1])
            end
        end
        
        #----------RESTRICCIONES DEL BESS-------
        # SOC DINÁMICO Y RESTRICCIONES DEL BESS
        @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
        for t in 2:T
            @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
            @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
            @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
            @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
            @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
        end
        
        # Penalización por SOC bajo
        for t in 1:T
            @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
        end
        
        #Evitar cargar y descargar el bess al mismo tiempo 
        for t in 1:T
            @constraint(model, charge[t] <= pev_max * (1 - z[t]))
            @constraint(model, discharge[t] <= pev_max * z[t])
        end
        
        #----------RESTRICCIONES DE REDES-------
        # LÍMITES DE FLUJO
        for (i,j) in keys(x), t in 1:T
            fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
            @constraint(model, fij <= Fmax[(i,j)])
            @constraint(model, fij >= -Fmax[(i,j)])
        end
        
        
        # ÁNGULO DE REFERENCIA
        for t in 1:T
            @constraint(model, θ[1,t] == 0)
        end
        
        # OBJETIVO
        # CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
        @constraint(model, soc[T] >= SOC_ini / 100 * Emax)

        @objective(model, Min,
            sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
            sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) +
            sum(pen_low[t]*pen_soc_low for t=1:T)
            + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
            + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
        )
        
        # OPTIMIZACIÓN
        tic = time()
        optimize!(model)
        toc = time()
        
        # RESULTADOS
        #status = termination_status(model)
        #println("\nEstado de optimización: ", status)
        println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

    if termination_status(model) == MOI.OPTIMAL || termination_status(model) == MOI.FEASIBLE_POINT
        return objective_value(model), bus_bess, bus_eolico;
    else
        return Inf, bus_bess, bus_eolico  # penalización por no factible
    end
end


mejor_costo = Inf
mejor_bess = 0
mejor_eolico = 0

for bess_nodo in 1:nBuses
    for eolico_nodo in 1:nBuses
        costo, b_bess, b_eolico = evaluar_modelo(bess_nodo, eolico_nodo)
        println("Costo: ", round(costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        if costo < mejor_costo
            mejor_costo = costo
            mejor_bess = b_bess
            mejor_eolico = b_eolico
            println("mejor_costo_new_encontrado: ", round(mejor_costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        end
    end
end

println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)


In [ ]:
println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)

In [ ]:
#-------------------------LIBRERIAS-----------------------------
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX

 
#-------------------------DATOS-----------------------------
        
# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
     
buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1

# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
        
# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])
        
T = length(demanda_horaria) # Longitud del horizonte temporal
        
        
# PARÁMETROS EÓLICOS
# Archivo Datos Eolicos
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
       
# Leer vectores por parque eólico
        
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2])
A_vec = Float64.(sheet_eolico["D"][2])
Cp_vec = Float64.(sheet_eolico["E"][2])


# PARÁMETROS VIENTO
# Archivo de Viento
        sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
        
        velocidades_viento = Float64.(sheet_viento["B"][2:end])
        function potencia_eolica_teorica(v)
            if v < 3 || v > 25
                return 0.0
            elseif v <= 11.8
                return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
            else
                return 1.5
            end
        end
        potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]
        
# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
        xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
        sheet_lineas = xlsx_lineas[1]
        
# Detectar cuántas líneas hay 
        nLineas = count(!ismissing, sheet_lineas["A"][2:end])
        
# Leer las columnas desde fila 2 hacia abajo
        desde = Int64.(sheet_lineas["A"][2:1+nLineas])
        hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
        reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
        flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
        
# Crear diccionarios para reactancia y Fmax
        x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
        Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
        Sbase = 100.0 # Potencia base del sistema en MVA
        
        nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema
        
# PARÁMETROS DE BESS
# Leer el archivo y hoja
        xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
        sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
        
        Emax             = Float64(sheet_bess["B"][2])
        pev_max          = Float64(sheet_bess["C"][2])
        η_c              = Float64(sheet_bess["D"][2])
        η_d              = Float64(sheet_bess["E"][2])
        ramp_pev         = Float64(sheet_bess["F"][2])
        SOC_ini          = Float64(sheet_bess["G"][2])
        SOC_min_pct      = Float64(sheet_bess["H"][2])
        pen_soc_low      = Float64(sheet_bess["I"][2])
        carga_bess       = Float64(sheet_bess["J"][2])
        descarga_bess    = Float64(sheet_bess["K"][2])
        degradacion_bess = Float64(sheet_bess["L"][2])
        
# DISTRIBUCION DE CARGA 
        xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
        sheet_carga = xlsx_carga[1]
        
        # Leer columnas
        buses_carga = Int64.(sheet_carga["A"][2:end])
        porcentajes_carga = Float64.(sheet_carga["B"][2:end])
        
        # Crear un diccionario para acceso rápido
        distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))        
        



function evaluar_modelo(bus_bess_actual::Int, bus_eolico_actual::Int)
        bus_eolico = bus_eolico_actual
        bus_bess         = bus_bess_actual        
        
        #-------------------------MODELO-----------------------------
        model = Model(HiGHS.Optimizer)
        
        
        #-------------------------VARIABLES-----------------------------
        # VARIABLES
        #@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
        @variable(model, p[g=1:nGen, t=1:T])
        @variable(model, u[g=1:nGen, t=1:T], Bin)
        @variable(model, v[g=1:nGen, t=1:T], Bin)
        @variable(model, θ[b=1:nBuses, t=1:T])
        @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
        @variable(model, 0 <= charge[t=1:T] <= pev_max)
        @variable(model, 0 <= discharge[t=1:T] <= pev_max)
        @variable(model, 0 <= soc[t=1:T] <= Emax)
        @variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
        @variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga
        
        
        #-------------------------RESTRICCIONES-----------------------------
        
        
        #----------RESTRICCIONES DE BALANCE DE NODOS-------
        # BALANCE NODAL
        for t in 1:T
            for b in 1:nBuses
                gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
                gen += (b == bus_eolico ? pw[t] : 0.0)
                gen += (b == bus_bess ? discharge[t] : 0.0)
                
                demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
                if b == bus_bess
                    demanda_b += charge[t]
                end
        
                inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
                outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        
                @constraint(model, gen + inflow - outflow == demanda_b)
            end
        end
        
        #----------RESTRICCIONES DE GENERADORES-------
        # RESTRICCIONES OPERATIVAS GENERADORES
        for g in 1:nGen, t in 1:T
            @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
            @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
        end
        
        # RESTRICCIONES DE ARRANQUE
        for g in 1:nGen
            @constraint(model, v[g,1] >= u[g,1] - u0[g])
            @constraint(model, v[g,1] <= u[g,1])
            @constraint(model, v[g,1] <= 1 - u0[g])
            for t in 2:T
                @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
                @constraint(model, v[g,t] <= u[g,t])
                @constraint(model, v[g,t] <= 1 - u[g,t-1])
            end
        end
        
        #----------RESTRICCIONES DEL BESS-------
        # SOC DINÁMICO Y RESTRICCIONES DEL BESS
        @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
        for t in 2:T
            @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
            @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
            @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
            @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
            @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
        end
        
        # Penalización por SOC bajo
        for t in 1:T
            @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
        end
        
        #Evitar cargar y descargar el bess al mismo tiempo 
        for t in 1:T
            @constraint(model, charge[t] <= pev_max * (1 - z[t]))
            @constraint(model, discharge[t] <= pev_max * z[t])
        end
        
        #----------RESTRICCIONES DE REDES-------
        # LÍMITES DE FLUJO
        for (i,j) in keys(x), t in 1:T
            fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
            @constraint(model, fij <= Fmax[(i,j)])
            @constraint(model, fij >= -Fmax[(i,j)])
        end
        
        
        # ÁNGULO DE REFERENCIA
        for t in 1:T
            @constraint(model, θ[1,t] == 0)
        end
        
        # OBJETIVO
        # CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
        @constraint(model, soc[T] >= SOC_ini / 100 * Emax)

        @objective(model, Min,
            sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
            sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) +
            sum(pen_low[t]*pen_soc_low for t=1:T)
            + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
            + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
        )
        
        # OPTIMIZACIÓN
        tic = time()
        optimize!(model)
        toc = time()
        
        # RESULTADOS
        #status = termination_status(model)
        #println("\nEstado de optimización: ", status)
        println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

    if termination_status(model) == MOI.OPTIMAL || termination_status(model) == MOI.FEASIBLE_POINT
        return objective_value(model), bus_bess, bus_eolico;
    else
        return Inf, bus_bess, bus_eolico  # penalización por no factible
    end
end


mejor_costo = 438777.42
mejor_bess = 6
mejor_eolico = 10

for bess_nodo in 7:nBuses
    for eolico_nodo in 1:nBuses
        costo, b_bess, b_eolico = evaluar_modelo(bess_nodo, eolico_nodo)
        println("Costo: ", round(costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        if costo < mejor_costo
            mejor_costo = costo
            mejor_bess = b_bess
            mejor_eolico = b_eolico
            println("mejor_costo_new_encontrado: ", round(mejor_costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        end
    end
end

println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)


In [ ]:
println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)

In [ ]:
#-------------------------LIBRERIAS-----------------------------
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX

 
#-------------------------DATOS-----------------------------
        
# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
     
buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1

# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
        
# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])
        
T = length(demanda_horaria) # Longitud del horizonte temporal
        
        
# PARÁMETROS EÓLICOS
# Archivo Datos Eolicos
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
       
# Leer vectores por parque eólico
        
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2])
A_vec = Float64.(sheet_eolico["D"][2])
Cp_vec = Float64.(sheet_eolico["E"][2])


# PARÁMETROS VIENTO
# Archivo de Viento
        sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
        
        velocidades_viento = Float64.(sheet_viento["B"][2:end])
        function potencia_eolica_teorica(v)
            if v < 3 || v > 25
                return 0.0
            elseif v <= 11.8
                return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
            else
                return 1.5
            end
        end
        potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]
        
# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
        xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
        sheet_lineas = xlsx_lineas[1]
        
# Detectar cuántas líneas hay 
        nLineas = count(!ismissing, sheet_lineas["A"][2:end])
        
# Leer las columnas desde fila 2 hacia abajo
        desde = Int64.(sheet_lineas["A"][2:1+nLineas])
        hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
        reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
        flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
        
# Crear diccionarios para reactancia y Fmax
        x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
        Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
        Sbase = 100.0 # Potencia base del sistema en MVA
        
        nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema
        
# PARÁMETROS DE BESS
# Leer el archivo y hoja
        xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
        sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
        
        Emax             = Float64(sheet_bess["B"][2])
        pev_max          = Float64(sheet_bess["C"][2])
        η_c              = Float64(sheet_bess["D"][2])
        η_d              = Float64(sheet_bess["E"][2])
        ramp_pev         = Float64(sheet_bess["F"][2])
        SOC_ini          = Float64(sheet_bess["G"][2])
        SOC_min_pct      = Float64(sheet_bess["H"][2])
        pen_soc_low      = Float64(sheet_bess["I"][2])
        carga_bess       = Float64(sheet_bess["J"][2])
        descarga_bess    = Float64(sheet_bess["K"][2])
        degradacion_bess = Float64(sheet_bess["L"][2])
        
# DISTRIBUCION DE CARGA 
        xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
        sheet_carga = xlsx_carga[1]
        
        # Leer columnas
        buses_carga = Int64.(sheet_carga["A"][2:end])
        porcentajes_carga = Float64.(sheet_carga["B"][2:end])
        
        # Crear un diccionario para acceso rápido
        distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))        
        



function evaluar_modelo(bus_bess_actual::Int, bus_eolico_actual::Int)
        bus_eolico = bus_eolico_actual
        bus_bess         = bus_bess_actual        
        
        #-------------------------MODELO-----------------------------
        model = Model(HiGHS.Optimizer)
        
        
        #-------------------------VARIABLES-----------------------------
        # VARIABLES
        #@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
        @variable(model, p[g=1:nGen, t=1:T])
        @variable(model, u[g=1:nGen, t=1:T], Bin)
        @variable(model, v[g=1:nGen, t=1:T], Bin)
        @variable(model, θ[b=1:nBuses, t=1:T])
        @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
        @variable(model, 0 <= charge[t=1:T] <= pev_max)
        @variable(model, 0 <= discharge[t=1:T] <= pev_max)
        @variable(model, 0 <= soc[t=1:T] <= Emax)
        @variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
        @variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga
        
        
        #-------------------------RESTRICCIONES-----------------------------
        
        
        #----------RESTRICCIONES DE BALANCE DE NODOS-------
        # BALANCE NODAL
        for t in 1:T
            for b in 1:nBuses
                gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
                gen += (b == bus_eolico ? pw[t] : 0.0)
                gen += (b == bus_bess ? discharge[t] : 0.0)
                
                demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
                if b == bus_bess
                    demanda_b += charge[t]
                end
        
                inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
                outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        
                @constraint(model, gen + inflow - outflow == demanda_b)
            end
        end
        
        #----------RESTRICCIONES DE GENERADORES-------
        # RESTRICCIONES OPERATIVAS GENERADORES
        for g in 1:nGen, t in 1:T
            @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
            @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
        end
        
        # RESTRICCIONES DE ARRANQUE
        for g in 1:nGen
            @constraint(model, v[g,1] >= u[g,1] - u0[g])
            @constraint(model, v[g,1] <= u[g,1])
            @constraint(model, v[g,1] <= 1 - u0[g])
            for t in 2:T
                @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
                @constraint(model, v[g,t] <= u[g,t])
                @constraint(model, v[g,t] <= 1 - u[g,t-1])
            end
        end
        
        #----------RESTRICCIONES DEL BESS-------
        # SOC DINÁMICO Y RESTRICCIONES DEL BESS
        @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
        for t in 2:T
            @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
            @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
            @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
            @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
            @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
        end
        
        # Penalización por SOC bajo
        for t in 1:T
            @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
        end
        
        #Evitar cargar y descargar el bess al mismo tiempo 
        for t in 1:T
            @constraint(model, charge[t] <= pev_max * (1 - z[t]))
            @constraint(model, discharge[t] <= pev_max * z[t])
        end
        
        #----------RESTRICCIONES DE REDES-------
        # LÍMITES DE FLUJO
        for (i,j) in keys(x), t in 1:T
            fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
            @constraint(model, fij <= Fmax[(i,j)])
            @constraint(model, fij >= -Fmax[(i,j)])
        end
        
        
        # ÁNGULO DE REFERENCIA
        for t in 1:T
            @constraint(model, θ[1,t] == 0)
        end
        
        # OBJETIVO
        # CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
        @constraint(model, soc[T] >= SOC_ini / 100 * Emax)

        @objective(model, Min,
            sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
            sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) +
            sum(pen_low[t]*pen_soc_low for t=1:T)
            + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
            + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
        )
        
        # OPTIMIZACIÓN
        tic = time()
        optimize!(model)
        toc = time()
        
        # RESULTADOS
        #status = termination_status(model)
        #println("\nEstado de optimización: ", status)
        println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

    if termination_status(model) == MOI.OPTIMAL || termination_status(model) == MOI.FEASIBLE_POINT
        return objective_value(model), bus_bess, bus_eolico;
    else
        return Inf, bus_bess, bus_eolico  # penalización por no factible
    end
end


mejor_costo = 438735.47
mejor_bess = 10
mejor_eolico = 10

for bess_nodo in 14:nBuses
    for eolico_nodo in 1:nBuses
        costo, b_bess, b_eolico = evaluar_modelo(bess_nodo, eolico_nodo)
        println("Costo: ", round(costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        if costo < mejor_costo
            mejor_costo = costo
            mejor_bess = b_bess
            mejor_eolico = b_eolico
            println("mejor_costo_new_encontrado: ", round(mejor_costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        end
    end
end

println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)


In [ ]:
println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)

In [ ]:
#-------------------------LIBRERIAS-----------------------------
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX

 
#-------------------------DATOS-----------------------------
        
# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
     
buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1

# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
        
# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])
        
T = length(demanda_horaria) # Longitud del horizonte temporal
        
        
# PARÁMETROS EÓLICOS
# Archivo Datos Eolicos
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
       
# Leer vectores por parque eólico
        
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2])
A_vec = Float64.(sheet_eolico["D"][2])
Cp_vec = Float64.(sheet_eolico["E"][2])


# PARÁMETROS VIENTO
# Archivo de Viento
        sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
        
        velocidades_viento = Float64.(sheet_viento["B"][2:end])
        function potencia_eolica_teorica(v)
            if v < 3 || v > 25
                return 0.0
            elseif v <= 11.8
                return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
            else
                return 1.5
            end
        end
        potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]
        
# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
        xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
        sheet_lineas = xlsx_lineas[1]
        
# Detectar cuántas líneas hay 
        nLineas = count(!ismissing, sheet_lineas["A"][2:end])
        
# Leer las columnas desde fila 2 hacia abajo
        desde = Int64.(sheet_lineas["A"][2:1+nLineas])
        hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
        reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
        flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
        
# Crear diccionarios para reactancia y Fmax
        x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
        Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
        Sbase = 100.0 # Potencia base del sistema en MVA
        
        nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema
        
# PARÁMETROS DE BESS
# Leer el archivo y hoja
        xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
        sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
        
        Emax             = Float64(sheet_bess["B"][2])
        pev_max          = Float64(sheet_bess["C"][2])
        η_c              = Float64(sheet_bess["D"][2])
        η_d              = Float64(sheet_bess["E"][2])
        ramp_pev         = Float64(sheet_bess["F"][2])
        SOC_ini          = Float64(sheet_bess["G"][2])
        SOC_min_pct      = Float64(sheet_bess["H"][2])
        pen_soc_low      = Float64(sheet_bess["I"][2])
        carga_bess       = Float64(sheet_bess["J"][2])
        descarga_bess    = Float64(sheet_bess["K"][2])
        degradacion_bess = Float64(sheet_bess["L"][2])
        
# DISTRIBUCION DE CARGA 
        xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
        sheet_carga = xlsx_carga[1]
        
        # Leer columnas
        buses_carga = Int64.(sheet_carga["A"][2:end])
        porcentajes_carga = Float64.(sheet_carga["B"][2:end])
        
        # Crear un diccionario para acceso rápido
        distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))        
        



function evaluar_modelo(bus_bess_actual::Int, bus_eolico_actual::Int)
        bus_eolico = bus_eolico_actual
        bus_bess         = bus_bess_actual        
        
        #-------------------------MODELO-----------------------------
        model = Model(HiGHS.Optimizer)
        
        
        #-------------------------VARIABLES-----------------------------
        # VARIABLES
        #@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
        @variable(model, p[g=1:nGen, t=1:T])
        @variable(model, u[g=1:nGen, t=1:T], Bin)
        @variable(model, v[g=1:nGen, t=1:T], Bin)
        @variable(model, θ[b=1:nBuses, t=1:T])
        @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
        @variable(model, 0 <= charge[t=1:T] <= pev_max)
        @variable(model, 0 <= discharge[t=1:T] <= pev_max)
        @variable(model, 0 <= soc[t=1:T] <= Emax)
        @variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
        @variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga
        
        
        #-------------------------RESTRICCIONES-----------------------------
        
        
        #----------RESTRICCIONES DE BALANCE DE NODOS-------
        # BALANCE NODAL
        for t in 1:T
            for b in 1:nBuses
                gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
                gen += (b == bus_eolico ? pw[t] : 0.0)
                gen += (b == bus_bess ? discharge[t] : 0.0)
                
                demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
                if b == bus_bess
                    demanda_b += charge[t]
                end
        
                inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
                outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        
                @constraint(model, gen + inflow - outflow == demanda_b)
            end
        end
        
        #----------RESTRICCIONES DE GENERADORES-------
        # RESTRICCIONES OPERATIVAS GENERADORES
        for g in 1:nGen, t in 1:T
            @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
            @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
        end
        
        # RESTRICCIONES DE ARRANQUE
        for g in 1:nGen
            @constraint(model, v[g,1] >= u[g,1] - u0[g])
            @constraint(model, v[g,1] <= u[g,1])
            @constraint(model, v[g,1] <= 1 - u0[g])
            for t in 2:T
                @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
                @constraint(model, v[g,t] <= u[g,t])
                @constraint(model, v[g,t] <= 1 - u[g,t-1])
            end
        end
        
        #----------RESTRICCIONES DEL BESS-------
        # SOC DINÁMICO Y RESTRICCIONES DEL BESS
        @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
        for t in 2:T
            @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
            @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
            @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
            @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
            @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
        end
        
        # Penalización por SOC bajo
        for t in 1:T
            @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
        end
        
        #Evitar cargar y descargar el bess al mismo tiempo 
        for t in 1:T
            @constraint(model, charge[t] <= pev_max * (1 - z[t]))
            @constraint(model, discharge[t] <= pev_max * z[t])
        end
        
        #----------RESTRICCIONES DE REDES-------
        # LÍMITES DE FLUJO
        for (i,j) in keys(x), t in 1:T
            fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
            @constraint(model, fij <= Fmax[(i,j)])
            @constraint(model, fij >= -Fmax[(i,j)])
        end
        
        
        # ÁNGULO DE REFERENCIA
        for t in 1:T
            @constraint(model, θ[1,t] == 0)
        end
        
        # OBJETIVO
        # CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
        @constraint(model, soc[T] >= SOC_ini / 100 * Emax)

        @objective(model, Min,
            sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
            sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) +
            sum(pen_low[t]*pen_soc_low for t=1:T)
            + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
            + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
        )
        
        # OPTIMIZACIÓN
        tic = time()
        optimize!(model)
        toc = time()
        
        # RESULTADOS
        #status = termination_status(model)
        #println("\nEstado de optimización: ", status)
        println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

    if termination_status(model) == MOI.OPTIMAL || termination_status(model) == MOI.FEASIBLE_POINT
        return objective_value(model), bus_bess, bus_eolico;
    else
        return Inf, bus_bess, bus_eolico  # penalización por no factible
    end
end


mejor_costo = Inf
mejor_bess = 0
mejor_eolico = 0

for bess_nodo in 1:nBuses
    for eolico_nodo in 1:nBuses
        costo, b_bess, b_eolico = evaluar_modelo(bess_nodo, eolico_nodo)
        println("Costo: ", round(costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        if costo < mejor_costo
            mejor_costo = costo
            mejor_bess = b_bess
            mejor_eolico = b_eolico
            println("mejor_costo_new_encontrado: ", round(mejor_costo, digits=2), " | BESS en: $b_bess | Eólico en: $b_eolico")
        end
    end
end

println("\n>> Mejor ubicación encontrada:")
println("Costo mínimo: ", round(mejor_costo, digits=2))
println("BESS en nodo: ", mejor_bess)
println("Eólico en nodo: ", mejor_eolico)
